<a href="https://colab.research.google.com/github/DaNa-cpu/parcele-timesen2crop/blob/main/IC3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conectare drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



Extragerea arhivei de date (unzip)


In [7]:
!unzip -o -q "/content/drive/MyDrive/Colab Notebooks/TimeSen2Crop.zip" -d "/content/TimeSen2Crop"

Detectarea structurii de directoare si fisiere

In [8]:
import os
import glob
import pandas as pd

base_dir = '/content/TimeSen2Crop'
all_csvs = glob.glob(os.path.join(base_dir, '**/*.csv'), recursive=True)
print(f'Total fisiere CSV identificate: {len(all_csvs)}')


Total fisiere CSV identificate: 1212240


Incarcarea pipeline-ului de metadate si date temporale

In [9]:
class TimeSen2CropPipeline:
    def __init__(self, base_dir):
        self.base_dir = base_dir
        self.dates_path = os.path.join(base_dir, 'TimeSen2Crop/33TUN/dates.csv')
        self.dates_df = None
        self.parcel_files = []

    def load_metadata(self):
        # Încarcă fișierul de date temporale
        if os.path.exists(self.dates_path):
            self.dates_df = pd.read_csv(self.dates_path)
            self.dates_df['acquisition_date'] = pd.to_datetime(self.dates_df['acquisition_date'].astype(str), format='%Y%m%d')
            print(f"[Pipeline] S-au încărcat {len(self.dates_df)} date de achiziție din dates.csv.")
        else:
            raise FileNotFoundError(f"Nu s-a găsit fișierul de date la: {self.dates_path}")

        # Selectează toate fișierele corespunzătoare parcelelor individuale din tile-ul 33TUN
        search_pattern = os.path.join(self.base_dir, 'TimeSen2Crop/33TUN/**/*.csv')
        all_files = glob.glob(search_pattern, recursive=True)
        self.parcel_files = [f for f in all_files if 'dates.csv' not in f]
        print(f"[Pipeline] S-au identificat exact {len(self.parcel_files)} parcele agricole individuale.")

    def load_parcel_data(self, index):
        """Încarcă datele spectrale ale unei singure parcele folosind indexul acesteia"""
        file_path = self.parcel_files[index]
        parcel_df = pd.read_csv(file_path)

        # Adăugăm axa temporală mapată direct peste observații
        if len(parcel_df) == len(self.dates_df):
            parcel_df['date'] = self.dates_df['acquisition_date']
        else:
            min_len = min(len(parcel_df), len(self.dates_df))
            parcel_df = parcel_df.iloc[:min_len].copy()
            parcel_df['date'] = self.dates_df['acquisition_date'].iloc[:min_len].values

        return parcel_df

# Pornim pipeline-ul pentru a inițializa metadatele în memorie
pipeline = TimeSen2CropPipeline('/content/TimeSen2Crop')
pipeline.load_metadata()

[Pipeline] S-au încărcat 35 date de achiziție din dates.csv.
[Pipeline] S-au identificat exact 31162 parcele agricole individuale.


Împărțirea setului de date (Train, Validation, Test Split)

In [10]:
import random

def split_dataset(parcel_files, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-9, "Suma procentelor trebuie să fie exact 1.0!"

    files = list(parcel_files)
    random.seed(seed)
    random.shuffle(files)

    total_files = len(files)
    train_end = int(total_files * train_ratio)
    val_end = train_end + int(total_files * val_ratio)

    train_set = files[:train_end]
    val_set = files[train_end:val_end]
    test_set = files[val_end:]

    print(f"Total parcele: {total_files}")
    print(f"  - Set Antrenare (Train):      {len(train_set)} parcele ({len(train_set)/total_files*100:.1f}%)")
    print(f"  - Set Validare (Validation): {len(val_set)} parcele ({len(val_set)/total_files*100:.1f}%)")
    print(f"  - Set Testare (Test):         {len(test_set)} parcele ({len(test_set)/total_files*100:.1f}%)")

    return train_set, val_set, test_set

train_files, val_files, test_files = split_dataset(pipeline.parcel_files)

Total parcele: 31162
  - Set Antrenare (Train):      21813 parcele (70.0%)
  - Set Validare (Validation): 4674 parcele (15.0%)
  - Set Testare (Test):         4675 parcele (15.0%)
